In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.2 Memory accounting

> 🎯 **Goal:** Estimate serving memory as weights plus activations plus KV cache, and see why the cache term is the one that runs you out of GPU first.

**Why this matters.** Before you deploy, you need to answer a blunt question: will this fit on the GPU I have, at the batch size and sequence length I want to serve. Guessing leads to out-of-memory crashes in production. A back-of-the-envelope memory budget tells you up front, and points you at which knob to turn (smaller model, fewer KV heads, fewer bits) when it does not fit.

**The intuition.** Think of GPU memory as a fixed budget you have to split three ways. The **weights** are a fixed cost, paid once no matter how many requests you serve. The **activations** are the scratch space for the current forward pass. The **KV cache** is the sneaky one: it is per-request and it grows the longer each conversation runs and the more conversations you serve at once. It behaves like a bill that scales with how busy you are, and it is usually what tips you over.

**The mechanics.** Some vocabulary. **Batch** is how many sequences you process together in one forward pass. The KV cache size is, per layer, two tensors (one K, one V) of shape (batch, n_head, seq, head_size), and you pay that for every layer. Multiply it out: 2 (K and V) times n_layer times n_head times seq times head_size times batch times bytes-per-number. In fp32 that is 4 bytes. The code below computes the weight memory (params times 4 bytes) and the KV cache for one batch and a 512-token sequence so you can compare the two terms directly. Notice that weights are constant while the cache term carries both `seq` and `batch` as linear factors.

In [ ]:
from pocketlm import PocketLM, PocketLMConfig

cfg = PocketLMConfig(vocab_size=100, block_size=512, n_layer=6, n_head=8, n_embd=512)
params = sum(p.numel() for p in PocketLM(cfg).parameters())
param_mb = params * 4 / 1e6                      # fp32 weights
head_size = cfg.n_embd // cfg.n_head
seq, batch = 512, 1
kv_mb = 2 * cfg.n_layer * cfg.n_head * seq * head_size * batch * 4 / 1e6
print(f"weights: {params / 1e6:.1f}M params = {param_mb:.1f} MB (fp32)")
print(f"KV cache @ seq={seq}, batch={batch}: {kv_mb:.2f} MB")

**What just happened.** The first line is your fixed weight cost in fp32. The second is the KV cache for a single 512-token request. Double the batch and that cache number doubles, double the sequence and it doubles again, because both appear as linear factors in the formula. The weight cost did not move. This is the whole reason serving optimizations target the cache: Grouped-Query Attention (fewer KV heads) and quantization (fewer bits per number) both shrink that term directly.

⚠️ **Common pitfalls**
- Budgeting only the weights and forgetting the cache, then crashing the moment real users send long prompts at high concurrency.
- Assuming fp32 everywhere. Most serving runs in fp16 or bf16 (2 bytes), which halves both terms, so always state the dtype with the number.
- Ignoring activations and framework overhead. This estimate is a floor, not a ceiling, leave headroom.

🏋️ **Try it yourself.** Change `batch` to 8 and `seq` to 2048 and rerun. Watch the cache term explode past the weight term. Then change the `4` (bytes) to `2` to model fp16 and confirm both numbers halve.

In [ ]:
assert abs(param_mb - params * 4 / 1e6) < 1e-9
assert kv_mb > 0